# T1D CGM baseline benchmark on Kaggle

This notebook trains the frozen two-feature CGM logistic comparator from a prepared Parquet table. It does not re-run the 19 GB Loop canonicalization audit. Upload the prepared private feature table and `loop_model_manifest.json` as a private Kaggle dataset.

Required columns: `patient_id`, `cohort_group`, `fold_id`, `label`, `current_glucose_mg_dl`, and `recent_slope_mg_dl_per_5m`. The locked holdout is never used during development-fold selection.

In [ ]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss

DATA_ROOT = Path('/kaggle/input/t1d-loop-prepared')
FEATURES = DATA_ROOT / 'loop_cgm_features.parquet'
MANIFEST = DATA_ROOT / 'loop_model_manifest.json'
assert FEATURES.exists(), f'Missing {FEATURES}; upload the prepared feature dataset first.'
assert MANIFEST.exists(), f'Missing {MANIFEST}'
print(FEATURES, FEATURES.stat().st_size / 1024**3, 'GiB')

In [ ]:
# Prefer GPU cuML when Kaggle exposes it; the fallback remains reproducible CPU sklearn.
try:
    import cudf
    from cuml.linear_model import LogisticRegression as GPULogisticRegression
    GPU = True
    print('Using cuDF/cuML GPU backend')
except Exception as exc:
    GPU = False
    from sklearn.linear_model import LogisticRegression
    print('Using sklearn CPU fallback:', type(exc).__name__)

In [ ]:
required = {'patient_id','cohort_group','fold_id','label','current_glucose_mg_dl','recent_slope_mg_dl_per_5m'}
# Read only the benchmark columns. For very large files, convert this cell to a Polars scan.
columns = sorted(required)
frame = cudf.read_parquet(FEATURES, columns=columns) if GPU else pd.read_parquet(FEATURES, columns=columns)
missing = required - set(frame.columns)
assert not missing, f'Missing columns: {missing}'
print('rows:', len(frame), 'columns:', list(frame.columns))
print(frame['cohort_group'].value_counts())

In [ ]:
with MANIFEST.open() as handle:
    manifest = json.load(handle)
assert manifest['version'] == 'loop-development-model-folds-v1'
folds = manifest['folds']
locked = set(manifest['locked_holdout_patients'])
assert len(locked) == 183
print('manifest participants:', len(manifest['patients']), 'locked holdout:', len(locked))

In [ ]:
feature_names = ['current_glucose_mg_dl', 'recent_slope_mg_dl_per_5m']
results = []
for fold in folds:
    fold_id = fold['fold_id']
    validation_id = 'outer-' + str((int(fold_id.split('-')[1]) + 1) % len(folds))
    # Development-only selection: train excludes this fold and the next validation fold.
    train_mask = (frame['cohort_group'] == 'development') & (frame['fold_id'] != fold_id) & (frame['fold_id'] != validation_id)
    val_mask = (frame['cohort_group'] == 'development') & (frame['fold_id'] == validation_id)
    X_train = frame.loc[train_mask, feature_names]
    y_train = frame.loc[train_mask, 'label']
    X_val = frame.loc[val_mask, feature_names]
    y_val = frame.loc[val_mask, 'label']
    if GPU:
        model = GPULogisticRegression(max_iter=1000, penalty='l2', C=1.0)
    else:
        model = LogisticRegression(max_iter=200, solver='lbfgs', n_jobs=-1)
    started = time.time()
    model.fit(X_train, y_train)
    score = model.predict_proba(X_val)[:, 1]
    y_np = y_val.to_numpy() if hasattr(y_val, 'to_numpy') else np.asarray(y_val)
    score_np = score.to_numpy() if hasattr(score, 'to_numpy') else (score.get() if hasattr(score, 'get') else np.asarray(score))
    results.append({'fold': fold_id, 'validation_fold': validation_id, 'windows': int(len(y_np)), 'positive_labels': int(y_np.sum()), 'average_precision': float(average_precision_score(y_np, score_np)), 'brier': float(brier_score_loss(y_np, score_np)), 'seconds': time.time() - started})
    print(results[-1])

In [ ]:
output = {'backend': 'cuml' if GPU else 'sklearn', 'features': feature_names, 'fold_results': results, 'mean_average_precision': float(np.mean([row['average_precision'] for row in results])), 'mean_brier': float(np.mean([row['brier'] for row in results]))}
print(json.dumps(output, indent=2))
Path('/kaggle/working/loop_cgm_logistic_development_results.json').write_text(json.dumps(output, indent=2) + '\n')

## Final holdout rule

Do not evaluate `locked_holdout` in this notebook while choosing features, epochs, learning rate, or model family. After the development pipeline is frozen, fit once on all development participants and evaluate the 183-person holdout in a separate, archived run.